# PRD-AGI Complete Ultimate Edition Setup
Run this notebook in Google Colab to start the unified system with all 10 advanced features.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/prd_agi_data/uploads
!mkdir -p /content/drive/MyDrive/prd_agi_data/checkpoints

In [ ]:
# Install Node.js
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs

# Ensure we are in the correct cloned repository
import os
import sys
# Look for git repos in content folder
dirs = [d for d in os.listdir('/content') if os.path.isdir(f'/content/{d}') and os.path.exists(f'/content/{d}/.git')]
if dirs:
    target_dir = f'/content/{dirs[0]}'
    os.chdir(target_dir)
    print(f"✅ Switched to Git Repository: {target_dir}")
else:
    print("⚠️ No Git repo found. Assuming working directly in /content")

# Install Python backend dependencies (Only if files exist)
if os.path.exists('prd-core/requirements.txt'):
    !pip install -r prd-core/requirements.txt
if os.path.exists('software-agents/requirements.txt'):
    !pip install -r software-agents/requirements.txt
if os.path.exists('private-llm/requirements.txt'):
    !pip install -r private-llm/requirements.txt

# Install React frontend dependencies
if os.path.exists('package.json'):
    !npm install
    !npm run postinstall || true


In [ ]:
# Set API keys from Colab secrets
import os
from google.colab import userdata

try:
   os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
   os.environ['NGROK_AUTH_TOKEN'] = userdata.get('NGROK_AUTH_TOKEN')
except:
   print('Please configure GEMINI_API_KEY and NGROK_AUTH_TOKEN in Colab Secrets.')

os.environ['SQLITE_PATH'] = '/content/drive/MyDrive/prd_agi_data/prd_llm.db'
os.environ['UPLOAD_DIR'] = '/content/drive/MyDrive/prd_agi_data/uploads/'


In [ ]:
# 1. Find correct project directory
import os
import sys
import time

print("📂 Finding correct working directory...")
dirs = [d for d in os.listdir('/content') if os.path.isdir(f'/content/{d}') and os.path.exists(f'/content/{d}/.git')]
if dirs:
    target_dir = f'/content/{dirs[0]}'
    os.chdir(target_dir)
    sys.path.append(target_dir)
    print(f"👉 Changed to: {target_dir}")

# 2. Clean up stale processes
print("🧹 Cleaning up old connections...")
!killall -9 ngrok node vite python3 2>/dev/null || true
time.sleep(2)

# 3. Pyngrok setup
!pip install pyngrok concurrently -q
from pyngrok import ngrok
try:
    ngrok.kill()
    # Provide fallback so it doesn't break
    token = os.environ.get('NGROK_AUTH_TOKEN', '3CejNGxKnVnTpANlpxcCg2kRJVs_5cnwGLDunSLajdPRN1Vgp')
    ngrok.set_auth_token(token)
    public_url = ngrok.connect(3000).public_url
    print('\n' + '='*50)
    print(f'🚀 PRD-AGI COMPLETE is live at: {public_url}')
    print('='*50 + '\n')
except Exception as e:
    print(f'\n❌ Ngrok error: {e}')

# 4. Boot the system
if os.path.exists('colab_start.py'):
    print("⏳ Starting application servers (Core, AI, DB)...")
    import subprocess
    subprocess.Popen('python colab_start.py', shell=True)
    time.sleep(3)
else:
    print("\n⚠️ Missing colab_start.py! Can't start background services.")

print("🚀 Booting frontend Vite UI on Port 3000...")
import subprocess
subprocess.Popen('npm run dev', shell=True)
